Bronze layer - ( raw data layer)

As i know the concept is to convert bronze means only the raw data without doing any modifications

In [0]:
df = spark.read.csv('/Workspace/Users/dehemiweerakoon@gmail.com/databricks-dubai-property-price/dubai_residential_data_2026.csv', header=True)


In [0]:
df.display()

In [0]:
df.write.format("delta").mode("overwrite").saveAsTable("bronze_data_dubai_residental_data")

Silver step

1.  Remove duplicates
2. fix or standardize data types
3. handle missing/null values
4. filter out clearly bad records

In [0]:
# to convert this data first to sliver first load bronze and then fix
bronze_data = spark.read.table("bronze_data_dubai_residental_data")
bronze_data.display()

In [0]:
standard_column_names = []
for rec in bronze_data.columns:
    standard_column_names.append(rec.lower())
standard_column_names


In [0]:
silver_df = bronze_data.toDF(*standard_column_names)
silver_df.display()

In [0]:
# I first checked whether need to change the data type
silver_df.dtypes

from their i see the instance_date , trans_value , actual_area and the price_per_sqm is not in correct data format i convert as following

In [0]:
from pyspark.sql.functions import col
silver_df = silver_df.withColumn('instance_date',col('instance_date').cast('date'))
silver_df = silver_df.withColumn('trans_value',col('trans_value').cast('double'))
silver_df = silver_df.withColumn('actual_area',col('actual_area').cast('double'))
silver_df = silver_df.withColumn('price_per_sqm',col('price_per_sqm').cast('double'))
silver_df.dtypes

In [0]:
silver_df.summary().toPandas()

In [0]:
# drop the duplciate data
print(silver_df.count())
silver_df = silver_df.dropDuplicates()
print(silver_df.count())


In [0]:
from pyspark.sql.functions import col, sum

# Loop through every column and count how many values are null
null_counts = silver_df.select([
    sum(col(c).isNull().cast("int")).alias(c) 
    for c in silver_df.columns
])

# Display the results as a neat table in Databricks
display(null_counts)

In here all the null value columns are string format not the double or the integer

In [0]:
silver_df = silver_df.fillna("Unknown", subset=["rooms_en", "nearest_mall_en", "nearest_landmark_en","project_en","nearest_metro_en"])

In [0]:
silver_df.summary().toPandas()

The number values are on  the range with noraml range so we dont need to fix those values

In [0]:
silver_df.write.format("delta").mode("overwrite").saveAsTable("silver_dubai_property")
display(silver_df.limit(10))

In [0]:
from pyspark.sql.types import BooleanType
import pyspark.sql.functions as F
silver_df = silver_df.withColumn(
    "is_free_hold_en",
    F.when(F.lower(F.col("is_free_hold_en")).contains("Non Free Hold"), False)
     .otherwise(True).cast(BooleanType())
)

In here I convert the column data into a standard format, In here i have converted the data types of the columns into the 'date' and 'double' when the necessary columns found. After that we drop duplicates in here. then count the number of null values in each column. but we found that null values are in the string columns only . then i replaced the null value with default values. the data ranges are normal the price and the numbers are in the range so we dont need to correct them and convert them in here. In the last the is_free_hold_en convert into boolean value telling true and false

# Gold

Gold: Aggregate your Silver table into the summary view(s) needed to answer your problem 
statement (e.g. totals, averages, trends over time, top-N breakdowns by category). 

### 1.  Gold 1: Avg price / price-per-sqft by district (area_en)

In [0]:
import pyspark.sql.functions as F

gold_1_df = silver_df.groupBy("area_en").agg(
    F.avg("actual_area").alias("avg_actual_area"),
    F.avg("trans_value").alias("avg_trans_value"),
    F.avg("price_per_sqm").alias("avg_price_per_sqm"),
    F.count("trans_value").alias("count_trans_value")
).orderBy(F.col("avg_price_per_sqm").desc())
display(gold_1_df)

###  Gold 2: Avg price / volume by property type & room count

In [0]:
gold_2_df = silver_df.groupBy('prop_sb_type_en','rooms_en').agg(
    F.avg("trans_value").alias("avg_trans_value"),
    F.avg("price_per_sqm").alias("avg_price_per_sqm"),
    F.count("trans_value").alias("count_trans_value")
).orderBy(F.col("avg_price_per_sqm").desc())
display(gold_2_df)


### Gold 3: Monthly transaction volume & price trend

In [0]:
gold_3_df = silver_df.withColumn('year_month',F.date_format('instance_date','yyyy-MM')).groupBy('year_month').agg(
                F.sum("trans_value").alias("sum_trans_value"),
                
                F.avg("price_per_sqm").alias("avg_price_per_sqm"),
                F.sum("price_per_sqm").alias("sum_price_per_sqm"),
                F.avg('trans_value').alias('avg_trans_value'))
display(gold_3_df)

In [0]:
gold_1_df.write.format("delta").mode("overwrite").saveAsTable("gold_price_by_area")
gold_2_df.write.format("delta").mode("overwrite").saveAsTable("gold_price_by_type")
gold_3_df.write.format("delta").mode("overwrite").saveAsTable("gold_monthly_view")